# Lesson 4 — Equilibrium: Finding Support Reactions

**Course:** Python for Structural Engineers — Simply Supported Beam  
**Prerequisites:** Lessons 1–3  
**You will learn:**
- Translate equilibrium equations directly into Python code
- Write a function that returns two values at once
- Add a self-check (assertion) to catch calculation errors automatically
- Build the complete beam+loads+reactions interactive widget

**Time estimate:** 60 minutes

---

## The Theory — Two Equations, Two Unknowns

A simply supported beam with two supports (pin A, roller B) is **statically determinate**. Two equilibrium equations are enough to find both reactions.

**Equation 1 — Sum of moments about A = 0:**

$$\sum M_A = 0: \quad R_B \cdot (x_B - x_A) + \sum_i P_i \cdot (x_i - x_A) = 0$$

Solving for R_B:

$$R_B = -\frac{\sum_i P_i \cdot (x_i - x_A)}{x_B - x_A}$$

**Equation 2 — Sum of vertical forces = 0:**

$$\sum F_y = 0: \quad R_A + R_B + \sum_i P_i = 0$$

Solving for R_A:

$$R_A = -\sum_i P_i - R_B$$

Where $P_i$ is each load (negative = downward) and $x_i$ is its position.

For a **UDL**, we replace it with an equivalent point load at its centroid:
$$P_{equiv} = w \cdot (x_2 - x_1), \qquad x_{centroid} = \frac{x_1 + x_2}{2}$$

---

## 4.1 From Equation to Code — Step by Step

In [ ]:
# Beam geometry
geometry = {"L_total": 10.0, "x_A": 2.0, "x_B": 8.0}

# Applied loads
loads = [
    {"type": "point", "x": 5.0,  "magnitude": -20.0},
    {"type": "udl",   "x1": 2.0, "x2": 8.0, "w": -5.0},
]

# ── Step 1: Extract geometry ───────────────────────────────────────────────
x_A  = geometry["x_A"]
x_B  = geometry["x_B"]
span = x_B - x_A

print(f"Support A at x = {x_A} m")
print(f"Support B at x = {x_B} m")
print(f"Interior span  = {span} m")

In [ ]:
# ── Step 2: Sum up all load contributions ─────────────────────────────────
total_force          = 0.0   # ΣPi  (will be negative for downward loads)
total_moment_about_A = 0.0   # ΣPi * (xi - xA)  (moment arm from support A)

for load in loads:
    if load["type"] == "point":
        P = load["magnitude"]
        x = load["x"]
        total_force          += P
        total_moment_about_A += P * (x - x_A)
        print(f"Point load: P = {P} kN at x = {x} m  →  moment arm = {x - x_A:.1f} m")

    elif load["type"] == "udl":
        w  = load["w"]
        x1, x2 = load["x1"], load["x2"]
        P_equiv   = w * (x2 - x1)          # equivalent point load
        x_centroid = (x1 + x2) / 2         # acts at midpoint of UDL
        total_force          += P_equiv
        total_moment_about_A += P_equiv * (x_centroid - x_A)
        print(f"UDL: w = {w} kN/m from {x1} to {x2} m  →  P_equiv = {P_equiv} kN at x_c = {x_centroid} m")

print(f"\nTotal vertical force  ΣPi = {total_force:.2f} kN")
print(f"Total moment about A  ΣMa = {total_moment_about_A:.2f} kN·m")

In [ ]:
# ── Step 3: Solve equilibrium equations ───────────────────────────────────
# ΣM_A = 0:  R_B * span + total_moment_about_A = 0
R_B = -total_moment_about_A / span

# ΣFy = 0:  R_A + R_B + total_force = 0
R_A = -total_force - R_B

print(f"Reaction at A:  R_A = {R_A:+.2f} kN  ({'upward' if R_A > 0 else 'downward'})")
print(f"Reaction at B:  R_B = {R_B:+.2f} kN  ({'upward' if R_B > 0 else 'downward'})")

In [ ]:
# ── Step 4: Self-check ────────────────────────────────────────────────────
# R_A + R_B + all loads must equal zero (ΣFy = 0)
residual = R_A + R_B + total_force
print(f"Equilibrium check  ΣFy = {residual:.6f} kN  (should be ≈ 0)")

# assert: if the condition is False, Python raises an error immediately
assert abs(residual) < 1e-6, f"Equilibrium not satisfied! Residual = {residual}"
print("✓ Equilibrium satisfied.")

---

## 4.2 Wrapping It in a Function That Returns Two Values

Functions can return more than one value — Python returns them as a **tuple**:

In [ ]:
def compute_reactions(geometry, loads):
    """Solve for support reactions using equilibrium.
    
    Returns R_A, R_B in kN (positive = upward).
    """
    x_A  = geometry["x_A"]
    x_B  = geometry["x_B"]
    span = x_B - x_A

    if span <= 0:
        raise ValueError("x_B must be greater than x_A.")

    total_force = 0.0
    total_M_A   = 0.0

    for load in loads:
        if load["type"] == "point":
            P = load["magnitude"]
            total_force += P
            total_M_A   += P * (load["x"] - x_A)
        elif load["type"] == "udl":
            P_eq = load["w"] * (load["x2"] - load["x1"])
            x_c  = (load["x1"] + load["x2"]) / 2
            total_force += P_eq
            total_M_A   += P_eq * (x_c - x_A)

    R_B = -total_M_A   / span
    R_A = -total_force - R_B

    # Built-in self-check
    assert abs(R_A + R_B + total_force) < 1e-6, "Equilibrium check failed!"

    return R_A, R_B   # Python lets you return two values


# Call the function — unpack both return values with comma syntax
R_A, R_B = compute_reactions(geometry, loads)

print(f"R_A = {R_A:+.2f} kN")
print(f"R_B = {R_B:+.2f} kN")

---

## 4.3 What Happens with Cantilever Loads?

Let's explore a key structural insight: a load on the **left cantilever** increases R_A but decreases R_B.

In [ ]:
geo = {"L_total": 10.0, "x_A": 2.0, "x_B": 8.0}

scenarios = [
    ("Load at midspan (x=5 m)",          [{"type": "point", "x": 5.0,  "magnitude": -10.0}]),
    ("Load on left cantilever (x=0 m)",  [{"type": "point", "x": 0.0,  "magnitude": -10.0}]),
    ("Load on right cantilever (x=10 m)",[{"type": "point", "x": 10.0, "magnitude": -10.0}]),
    ("Load directly over A (x=2 m)",     [{"type": "point", "x": 2.0,  "magnitude": -10.0}]),
]

print(f"{'Scenario':<40}  {'R_A':>8}  {'R_B':>8}  {'Sum':>8}")
print("-" * 68)
for name, loads_s in scenarios:
    Ra, Rb = compute_reactions(geo, loads_s)
    print(f"{name:<40}  {Ra:>+8.2f}  {Rb:>+8.2f}  {Ra+Rb:>+8.2f} kN")

**Observations:**
- A load directly over A → all load goes to A (R_A = −load, R_B = 0)
- A load on the left cantilever → R_B becomes **negative** (pulls the roller down)
- The sum R_A + R_B always equals the total applied load in magnitude

---

## 4.4 Full Interactive Widget — Beam + Loads + Reactions

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import sys
sys.path.insert(0, "../shared")
from plot_helpers import draw_beam, draw_loads, draw_reactions

%matplotlib inline

# ── Shared state ──────────────────────────────────────────────────────────
live_loads = []
out_plot   = widgets.Output()
out_status = widgets.Output()

# ── Geometry controls ─────────────────────────────────────────────────────
style  = {"description_width": "170px"}
layout = widgets.Layout(width="460px")

sl_L  = widgets.FloatSlider(value=10.0, min=5.0, max=20.0, step=0.5,
                             description="Total length L (m):", style=style, layout=layout)
sl_xA = widgets.FloatSlider(value=2.0,  min=0.0, max=9.5,  step=0.5,
                             description="Support A at (m):",    style=style, layout=layout)
sl_xB = widgets.FloatSlider(value=8.0,  min=0.5, max=10.0, step=0.5,
                             description="Support B at (m):",    style=style, layout=layout)

# ── Load controls ─────────────────────────────────────────────────────────
dd_type = widgets.Dropdown(options=[("Point load", "point"), ("UDL", "udl")],
                            value="point", description="Load type:", style=style, layout=layout)
sl_x    = widgets.FloatSlider(value=5.0, min=0.0, max=10.0, step=0.5,
                               description="Position x (m):", style=style, layout=layout)
sl_P    = widgets.FloatSlider(value=-20.0, min=-150.0, max=150.0, step=5.0,
                               description="Magnitude (kN):", style=style, layout=layout)
sl_x1   = widgets.FloatSlider(value=2.0, min=0.0, max=9.5, step=0.5,
                               description="UDL start x1 (m):", style=style, layout=layout)
sl_x2   = widgets.FloatSlider(value=8.0, min=0.5, max=10.0, step=0.5,
                               description="UDL end x2 (m):",   style=style, layout=layout)
sl_w    = widgets.FloatSlider(value=-5.0, min=-40.0, max=40.0, step=1.0,
                               description="Intensity (kN/m):", style=style, layout=layout)

btn_add   = widgets.Button(description="Add Load",  button_style="primary")
btn_clear = widgets.Button(description="Clear All", button_style="danger")

pt_ctrl  = widgets.VBox([sl_x, sl_P])
udl_ctrl = widgets.VBox([sl_x1, sl_x2, sl_w])
udl_ctrl.layout.display = "none"

def on_type(change):
    if change["new"] == "point":
        pt_ctrl.layout.display  = ""
        udl_ctrl.layout.display = "none"
    else:
        pt_ctrl.layout.display  = "none"
        udl_ctrl.layout.display = ""
dd_type.observe(on_type, names="value")

# ── Main draw function ────────────────────────────────────────────────────
def redraw(change=None):
    L   = sl_L.value
    x_A = min(sl_xA.value, sl_xB.value - 0.5)
    x_B = min(sl_xB.value, L)
    geo = {"L_total": L, "x_A": x_A, "x_B": x_B}

    with out_plot:
        out_plot.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(13, 4.5))
        draw_beam(ax, geo)
        draw_loads(ax, live_loads, L)

        if live_loads:
            try:
                R_A, R_B = compute_reactions(geo, live_loads)
                draw_reactions(ax, x_A, x_B, R_A, R_B)
                ax.set_title(
                    f"R_A = {R_A:+.2f} kN   |   R_B = {R_B:+.2f} kN",
                    fontsize=11, pad=8
                )
            except Exception as e:
                ax.set_title(f"Error: {e}", fontsize=10, color="red")
        else:
            ax.set_title("Add loads to see reactions", fontsize=10, color="gray")

        plt.tight_layout()
        plt.show()

# Connect geometry sliders
for sl in [sl_L, sl_xA, sl_xB]:
    sl.observe(redraw, names="value")

# Button callbacks
def on_add(b):
    L = sl_L.value
    with out_status:
        out_status.clear_output(wait=True)
        if dd_type.value == "point":
            live_loads.append({"type": "point", "x": sl_x.value, "magnitude": sl_P.value})
            print(f"✓ Added {sl_P.value:+.0f} kN at x = {sl_x.value:.1f} m")
        else:
            if sl_x2.value <= sl_x1.value:
                print("Error: x2 must be > x1")
                return
            live_loads.append({"type": "udl", "x1": sl_x1.value,
                                "x2": sl_x2.value, "w": sl_w.value})
            print(f"✓ Added UDL {sl_w.value:+.0f} kN/m from {sl_x1.value:.1f} to {sl_x2.value:.1f} m")
    redraw()

def on_clear(b):
    live_loads.clear()
    with out_status:
        out_status.clear_output(wait=True)
        print("Loads cleared.")
    redraw()

btn_add.on_click(on_add)
btn_clear.on_click(on_clear)

# ── Initial render ────────────────────────────────────────────────────────
redraw()
display(widgets.VBox([
    widgets.HTML("<b>Geometry</b>"),
    sl_L, sl_xA, sl_xB,
    widgets.HTML("<hr><b>Add Loads</b>"),
    dd_type, pt_ctrl, udl_ctrl,
    widgets.HBox([btn_add, btn_clear]),
    out_status,
    out_plot
]))

**Experiments to try:**
1. Add a `−20 kN` load at x = 5 m. Move support A left and right — watch how R_A and R_B change.
2. Add a load at x = 0 m (left tip). Notice R_B goes **negative** — the roller must be anchored.
3. Add a symmetric UDL and place the supports symmetrically — R_A should equal R_B.

---

## 4.5 Practice Exercise

Using `compute_reactions()`, find R_A and R_B for:
- Beam: L = 10 m, x_A = 2 m, x_B = 8 m  
- Loads: −30 kN at x = 3 m, −10 kN/m UDL from 4 m to 10 m

Verify manually that R_A + R_B = total applied load.

In [ ]:
# Your code here


---

## Summary

**Python concepts covered:**
- Functions returning multiple values: `return R_A, R_B`
- Unpacking: `R_A, R_B = compute_reactions(...)`
- `assert` — raising an error if a condition is not met
- `raise ValueError(...)` — deliberately triggering an error with a message

**Structural concepts covered:**
- Static equilibrium: ΣFy = 0 and ΣM = 0
- UDL to equivalent point load conversion
- Effect of cantilever loads on reaction signs

## Before the Next Lesson

In **Lesson 5** you'll use the reactions to build the shear force diagram (SFD). Think about this: if you cut the beam at any section, the shear force is just the sum of everything to the left of the cut. We'll automate that sum for 500 points.